# Mobile Push Notification Targeting Model

This analysis addresses a targeting optimization problem for Tuango, a mobile deal platform in China. The goal is to determine which customers should receive a push notification for a karaoke deal — maximizing campaign profitability by targeting high-intent users rather than broadcasting to everyone.

**Business context:** Sending push notifications is not free. Over-messaging customers increases the probability they block future notifications, permanently closing a marketing channel. The true marginal cost of each message is estimated at 9 RMB.

**Approach:** We use a 5% pilot sample (20,908 customers) to build logistic and linear regression models, then apply them to determine the optimal targeting strategy for the remaining 397,252 customers.

In [ ]:
import polars as pl
import pyrsm as rsm
import pandas as pd
from plotnine import ggplot, aes, geom_col, geom_text, labs, theme_minimal, theme_bw, scale_y_continuous

In [ ]:
tuango = pl.read_parquet("data/tuango_pre.parquet")

## Data Preparation

The  column contains categorical values ("yes"/"no") with nulls for customers in the rollout group who have not yet received the offer. We create a binary numeric variable  for modeling.

In [ ]:
# create a variable called 'buyer_yes' that has value 1 when buyer == 'yes', has value 0 when buyer == 'no' and has value np.nan when buyer.isna() is True
tuango = tuango.with_columns(
    pl.when(pl.col("buyer") == "yes")
    .then(1)
    .when(pl.col("buyer") == "no")
    .then(0)
    .otherwise(None)
    .alias("buyer_yes")
)


# you can check that you have this set up correctly using the command below
tuango.select(pl.col("buyer_yes").value_counts())

## Part I: Response Rate Analysis

### Baseline Purchase Rate

What proportion of customers in the pilot sample responded to the deal offer?

In [ ]:
test_data = tuango.filter(pl.col("buyer").is_not_null())

q1 = test_data.filter(pl.col("buyer") == "yes").height / test_data.height


### Order Size Distribution

Among customers who purchased, what was the average number of karaoke sessions ordered?

In [ ]:
q2 = tuango.group_by("buyer").agg(
    pl.col("ordersize").count().alias("count"),
    pl.col("ordersize").mean().alias("mean"),
    pl.col("ordersize").std().alias("std"),
)

q2


## Part II: Building Targeting Models

### Logistic Regression — Predicting Purchase Probability

We model the probability that a customer buys the deal using RFM variables (recency, frequency, monetary) alongside age, gender, and music category history.

In [ ]:
clf = rsm.model.logistic(
    data={"tuango": tuango},
    rvar="buyer",
    lev="yes",
    evar=["recency", "frequency", "monetary", "age", "music", "gender"],
)

clf.summary()


### Partial Dependence Plots

PDPs show the marginal effect of each predictor on purchase probability, holding all other variables constant.

In [ ]:
clf.plot(
    "pdp",
    incl=["recency", "frequency", "monetary", "age", "music", "gender"],
    minq=0,
    maxq=1,
)


The probability of buying the karaoke package becomes lower with recency, though slightly, and rises with frequency. The monetary value has the strongest positive impact, and the more the customers spend, the higher their chances of buying the package. The age factor has a non-linear relationship, and the younger and middle-aged customers have higher chances of buying than the older ones. The gender factor has a slight effect. Music-loving customers have higher chances of buying the package.

### Permutation Importance — Logistic Regression

Permutation importance measures how much model performance (AUC) drops when each variable is randomly shuffled — breaking its relationship with the outcome. Higher drop = more important variable.

In [ ]:
clf.plot("pip")

Permutation Importance quantifies the change in model performance caused by randomly shuffling the values of a variable, breaking its association with the result. Variables that result in a higher decline in AUC following permutation are deemed more essential. In this model, **MOST IMPORTANT** causes the greatest decline in AUC and hence is the most important predictor, whereas **LEAST IMPORTANT** causes the lowest performance change and looks to be the least important variable.

### Model Calibration Check

When predictions are made on the same data used for estimation, the average predicted probability should equal the observed response rate — a basic calibration check.

In [ ]:
tuango = tuango.with_columns(pred_logit=clf.predict(tuango).get_column("prediction"))

tuango.filter((tuango["test"] == 1)).select(
    pl.col("buyer_yes").mean(), pl.col("pred_logit").mean()
)


When you use the same data that was used to estimate the logistic regression, the average predicted probability will match the observed response rate. This happens because the intercept is set to align with the mean outcome from the sample, which is more about calibration than it is about predictive accuracy.

### Linear Regression — Predicting Order Size

For customers who purchase, we model how many sessions they are likely to buy. This is estimated only on buyers, since order size is unobserved for non-buyers.

In [ ]:
reg = rsm.model.regress(
    data={"tuango": tuango.filter((tuango["buyer_yes"] == 1) & (tuango["test"] == 1))},
    rvar="ordersize",
    evar=["recency", "frequency", "monetary", "age", "gender", "music"],
)

reg.summary()

It makes sense to focus on customers who placed an order because order size is only observed for buyers. Restricting the sample allows the regression to isolate factors that affect how much customers spend, conditional on making a purchase. However, the results are conditional on purchase and may be subject to selection bias, so they do not explain the purchase decision itself.



### Permutation Importance — Linear Regression

In [ ]:
reg.plot("pip")

The permutation importance plot indicates that monetary is the strongest predictor in the model, with frequency ranking second.When these variables are randomly shuffled, the model’s performance falls the most, which suggests the model depends strongly on them.The other variables of recency, age, gender, and music, seem to matter less, and they add only a small amount to how well the model predicts outcomes.

The outcomes of the linear regression indicate that our capacity to forecast the order size for customers who accepted the deal is highly restricted.The model has a very low $R^2$  (approximately 0.003), indicating that it explains almost none of the variation in order size. In addition, most explanatory variables are not statistically significant, further suggesting weak predictive power.

### Predicted vs Actual Order Size

In [ ]:
tuango = tuango.with_columns(pred_reg=reg.predict(tuango).get_column("prediction"))


tuango.filter((pl.col("test") == 1) & (pl.col("buyer") == "yes")).select(
    pl.col("ordersize").mean(), pl.col("pred_reg").mean()
)


Among buyers, the average predicted order size from the linear regression is exactly equal to the average observed order size, but this does not imply good predictive performance.

## Part III: Profitability Analysis

**Campaign assumptions:**
- Price per 30-minute session: 49 RMB
- Marginal cost per message: 9 RMB
- Tuango revenue share: 50% of sales

### Breakeven Response Rate

The minimum response rate required for a message to be profitable, given the margin structure.

In [ ]:
# the breakeven response rate

cost = 9.0  # float, marginal cost per message (RMB)

avg_ordersize_q2 = 3.941089  # float, average ordersize from Question 2

price = 49.0  # float, RMB per 30-min session
fee = 0.5  # float, Tuango fee share of sales revenues

margin = fee * price * avg_ordersize_q2  # float

breakeven = cost / margin  # float
q11 = breakeven

q11


### Strategy 1: Broadcast (Target Everyone)

In [ ]:
avg_ordersize = (
    tuango.filter((pl.col("buyer") == "yes") & (pl.col("test") == 1))
    .select(pl.col("ordersize").mean())
    .item()
)
margin = 0.5 * 49 * avg_ordersize  # num*50%*49


In [ ]:
nr_message_all = 397_252
message_cost = 9

response_rate = (
    tuango.filter(pl.col("test") == 1).select((pl.col("buyer") == "yes").mean()).item()
)

nr_responses_all = nr_message_all * response_rate
revenue_all = nr_responses_all * 49 * avg_ordersize
message_cost_all = nr_message_all * message_cost

profit_all = 0.5 * revenue_all - message_cost_all
rome_all = profit_all / message_cost_all

print(f"Projected profit (RMB): {profit_all:,.2f}")
print(f"Return on marketing expenditures (ROME): {rome_all:.4f}")


Offering the deal to all 397,252 remaining customers is expected to generate a projected profit of approximately **130,578 RMB**, with a return on marketing expenditures (ROME) of about **3.65%**.


### Strategy 2: Model-Based Targeting

Only message customers whose predicted purchase probability exceeds the breakeven rate.

In [ ]:
breakeven_rate = 9 / (0.5 * 49 * avg_ordersize)
print("=== Q13: Targeted Campaign ===")
print(f"Breakeven response rate: {breakeven_rate:.4f}")

In [ ]:
targeted_rollout = tuango.filter(
    (pl.col("test") == 0) & (pl.col("pred_logit") > breakeven_rate)
)

nr_message_targeted = targeted_rollout.height
print(f"Number of messages sent (targeted): {nr_message_targeted:,}")


In [ ]:
# targeted response rate (in test sample)
response_rate_targeted = (
    tuango.filter(pl.col("test") == 1)
    .filter(pl.col("pred_logit") > breakeven_rate)
    .select((pl.col("buyer") == "yes").mean())
    .item()
)

# fine-tuned avg order size: among targeted people who bought
avg_ordersize_targeted = (
    tuango.filter(pl.col("test") == 1)
    .filter(pl.col("pred_logit") > breakeven_rate)
    .filter(pl.col("buyer") == "yes")
    .select(pl.col("ordersize").mean())
    .item()
)

print(f"Response rate (targeted, test sample): {response_rate_targeted:.4f}")
print(f"Avg order size (targeted buyers): {avg_ordersize_targeted:.4f}")


In [ ]:
price = 49
fee_share = 0.5
message_cost = 9

nr_responses_targeted = nr_message_targeted * response_rate_targeted
revenue_targeted = nr_responses_targeted * price * avg_ordersize_targeted
message_cost_targeted = nr_message_targeted * message_cost

profit_targeted = fee_share * revenue_targeted - message_cost_targeted
rome_targeted = profit_targeted / message_cost_targeted

print(f"Projected profit (RMB): {profit_targeted:,.2f}")
print(f"ROME: {rome_targeted:.4f}")

By targeting only those customers who have a predicted probability of purchase above the breakeven response rate (0.0932), the number of messages being sent out comes down to around 174,671 customers. The targeted approach shows a predicted profit of around RMB 884,566 with a return on marketing expenditure (ROME) of about 0.56. In contrast to the previous approach in Question 12, where all customers were targeted (profit ≈ RMB 130,578, ROME ≈ 0.04), the current approach sends out fewer messages and is much more profitable.

### Profit Comparison

In [ ]:
import pandas as pd
from plotnine import (
    ggplot,
    aes,
    geom_col,
    geom_text,
    labs,
    theme_minimal,
    scale_y_continuous,
)

profit_df = pd.DataFrame(
    {
        "Strategy": ["Target Everyone", "Targeted"],
        "Profit": [profit_all, profit_targeted],
    }
)


In [ ]:
(
    ggplot(profit_df, aes(x="Strategy", y="Profit"))
    + geom_col()
    + geom_text(aes(label="Profit"), va="bottom", format_string="{:,.0f}", size=10)
    + scale_y_continuous(labels=lambda l: [f"{int(x):,}" for x in l])
    + labs(
        title="Projected Profit Comparison (Q12 vs Q13)",
        x="Campaign Strategy",
        y="Projected Profit (RMB)",
    )
    + theme_minimal()
)


The bar chart shows that targeting all customers (Question 12) results in a projected profit of approximately RMB 130,578, while the targeted strategy (Question 13) generates a substantially higher projected profit of approximately RMB 884,566. This comparison indicates that the targeted campaign is significantly more profitable than targeting everyone.

### ROME Comparison

In [ ]:
import pandas as pd
from plotnine import ggplot, aes, geom_col, geom_text, labs, theme_minimal

rome_df = pd.DataFrame(
    {
        "Strategy": ["Target Everyone", "Targeted"],
        "ROME_pct": [rome_all * 100, rome_targeted * 100],
    }
)

rome_df


In [ ]:
(
    ggplot(rome_df, aes(x="Strategy", y="ROME_pct"))
    + geom_col()
    + geom_text(aes(label="ROME_pct"), va="bottom", format_string="{:.1f}%", size=10)
    + labs(
        title="Return on Marketing Expenditures (Q12 vs Q13)",
        x="Campaign Strategy",
        y="ROME (%)",
    )
    + theme_minimal()
)


## Part IV: Validation on Actual Rollout Data

Tuango sent the offer to all 397,252 remaining customers to generate validation data. We now evaluate how the targeting strategy performed against actual outcomes — not projections.

In [ ]:
import polars as pl

tuango_post = pl.read_parquet("data/tuango_post.parquet")


In [ ]:
# Predict on post data using the pre-trained model
pred_post = clf.predict(tuango_post)

# Extract prediction column and attach to tuango_post
pred_post_df = pl.DataFrame(pred_post).select("prediction")

tuango_post = tuango_post.hstack(pred_post_df.rename({"prediction": "pred_logit"}))

# Sanity check
tuango_post.select(pl.col("pred_logit").head())


In [ ]:
avg_ordersize_test = (
    tuango_post.filter((pl.col("test") == 1) & (pl.col("buyer") == "yes"))
    .select(pl.col("ordersize").mean())
    .item()
)

breakeven_rate_post = 9 / (0.5 * 49 * avg_ordersize_test)
breakeven_rate_post


In [ ]:
rollout_targeted = tuango_post.filter(
    (pl.col("test") == 0) & (pl.col("pred_logit") > breakeven_rate_post)
)

n_msg_targeted_post = rollout_targeted.height

revenue_targeted_post = (
    rollout_targeted.filter(pl.col("buyer") == "yes")
    .select((pl.col("ordersize") * 49).sum())
    .item()
)

profit_targeted_post = 0.5 * revenue_targeted_post - n_msg_targeted_post * 9
rome_targeted_post = profit_targeted_post / (n_msg_targeted_post * 9)

profit_targeted_post, rome_targeted_post


In [ ]:
# Actual outcomes: No targeting (roll-out sample, test = 0)

rollout_all = tuango_post.filter(pl.col("test") == 0)

n_msg_all_post = rollout_all.height

revenue_all_post = (
    rollout_all.filter(pl.col("buyer") == "yes")
    .select((pl.col("ordersize") * 49).sum())
    .item()
)

profit_all_post = 0.5 * revenue_all_post - n_msg_all_post * 9
rome_all_post = profit_all_post / (n_msg_all_post * 9)


In [ ]:
performance_profit_pl = pl.DataFrame(
    {
        "Strategy": ["No targeting (actual)", "Logit targeting (actual)"],
        "Profit": [profit_all, profit_targeted_post],
    }
)

performance_profit_pd = performance_profit_pl.to_pandas()


In [ ]:
from plotnine import ggplot, aes, geom_col, geom_text, labs, theme_bw

(
    ggplot(performance_profit_pd, aes(x="Strategy", y="Profit"))
    + geom_col()
    + geom_text(aes(label="Profit"), format_string="{:,.0f}", va="bottom")
    + labs(title="Actual Profit on Roll-out Sample (test = 0)", x="", y="Profit (RMB)")
    + theme_bw()
)


In [ ]:
# Q16–15: ROME comparison (actual outcomes)
performance_rome_pl = pl.DataFrame(
    {
        "Strategy": ["No targeting (actual)", "Logit targeting (actual)"],
        "ROME": [rome_all_post * 100, rome_targeted_post * 100],
    }
)

performance_rome_pd = performance_rome_pl.to_pandas()

from plotnine import ggplot, aes, geom_col, geom_text, labs, theme_bw

(
    ggplot(performance_rome_pd, aes(x="Strategy", y="ROME"))
    + geom_col()
    + geom_text(aes(label="ROME"), format_string="{:.1f}%", va="bottom")
    + labs(title="Actual ROME on Roll-out Sample (test = 0)", x="", y="ROME (%)")
    + theme_bw()
)
